# Tool Use 기초 (5) — 내장 도구 (Built-in Tools)

**Skilljar Lessons 10-11 대응**

이 노트북에서 다루는 내용:
1. Text Editor Tool — 파일 생성/읽기/편집
2. Web Search Tool — 웹 검색 및 인용
3. 응답 블록 처리 (ServerToolUseBlock, WebSearchToolResultBlock, Citations)

**Built-in Tools**는 사용자가 함수를 구현할 필요 없이,  
Claude가 서버 측에서 직접 실행하는 도구입니다.

In [1]:
# ── Setup ──────────────────────────────────────────────
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

---

## §1. Text Editor Tool — 스키마 정의

Text Editor Tool은 Claude가 파일을 **생성, 읽기, 편집**할 수 있게 합니다.  
일반 tool과 달리 `type: "text_editor_20250124"`로 지정합니다.

> **주의**: 모델에 따라 스키마 버전이 다릅니다:
> - Claude 3.5 Sonnet: `text_editor_20241022`
> - Claude 3.7+ / Claude 4+: `text_editor_20250124`

In [2]:
# Text Editor Tool 스키마
# Claude 4 계열 (Haiku 4.5 포함)
text_editor_tool = {
    "type": "text_editor_20250124",
    "name": "str_replace_based_edit_tool",
}

# 참고: Claude 3.5 Sonnet 사용 시
# text_editor_tool = {
#     "type": "text_editor_20241022",
#     "name": "str_replace_editor",
# }

print("Text Editor Tool schema:")
print(f"  type: {text_editor_tool['type']}")
print(f"  name: {text_editor_tool['name']}")

Text Editor Tool schema:
  type: text_editor_20250124
  name: str_replace_based_edit_tool


## §2. Text Editor Tool — 사용 예제

Text Editor Tool은 다음 명령(command)을 지원합니다:
- `view` — 파일 내용 읽기
- `create` — 새 파일 생성
- `str_replace` — 문자열 치환으로 편집
- `insert` — 특정 줄에 텍스트 삽입

In [3]:
# 파일 생성 요청
response = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    tools=[text_editor_tool],
    messages=[
        {
            "role": "user",
            "content": (
                "Create a Python file called '/tmp/hello.py' that prints "
                "'Hello from Claude!' and includes a function that adds two numbers."
            ),
        }
    ],
)

print("stop_reason:", response.stop_reason)
print()
for block in response.content:
    print(f"Block type: {block.type}")
    if hasattr(block, "text"):
        print(f"  text: {block.text}")
    elif hasattr(block, "name"):
        print(f"  name: {block.name}")
        print(f"  input: {block.input}")
    print()

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': "tools.0.text_editor_20250124.name: Input should be 'str_replace_editor'"}, 'request_id': 'req_011Ca36woQ8rGyCHTfphcpqV'}

In [ ]:
# 파일 읽기 요청
response_view = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    tools=[text_editor_tool],
    messages=[
        {
            "role": "user",
            "content": "View the contents of '/tmp/hello.py'",
        }
    ],
)

print("stop_reason:", response_view.stop_reason)
print()
for block in response_view.content:
    print(f"Block type: {block.type}")
    if hasattr(block, "input"):
        print(f"  command: {block.input.get('command', 'N/A')}")
        print(f"  path: {block.input.get('path', 'N/A')}")

---

## §3. Web Search Tool — 스키마 정의

Web Search Tool은 Claude가 **실시간 웹 검색**을 수행하여  
최신 정보를 기반으로 답변할 수 있게 합니다.

스키마에는 `type`, `name`, 그리고 선택적으로 `max_uses`를 지정합니다.

In [ ]:
# Web Search Tool 스키마
web_search_tool = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,  # 한 번의 응답에서 최대 검색 횟수
}

print("Web Search Tool schema:")
print(f"  type:     {web_search_tool['type']}")
print(f"  name:     {web_search_tool['name']}")
print(f"  max_uses: {web_search_tool['max_uses']}")

## §4. Web Search Tool — 기본 검색 (Basic Query)

Claude에게 최신 정보가 필요한 질문을 하면,  
자동으로 웹 검색을 수행합니다.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    tools=[web_search_tool],
    messages=[
        {
            "role": "user",
            "content": "What is the latest version of Python released in 2025?",
        }
    ],
)

print("stop_reason:", response.stop_reason)
print(f"Content blocks: {len(response.content)}")
print()

# 최종 텍스트 출력
for block in response.content:
    if hasattr(block, "text"):
        print("Claude:", block.text[:500])
        break

## §5. Web Search Tool — 도메인 제한 (Allowed Domains)

`allowed_domains`로 검색 범위를 특정 웹사이트로 제한할 수 있습니다.  
이는 신뢰할 수 있는 소스만 사용하고 싶을 때 유용합니다.

In [ ]:
# 도메인 제한 검색
web_search_restricted = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
    "allowed_domains": ["docs.anthropic.com", "github.com"],
}

response_restricted = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    tools=[web_search_restricted],
    messages=[
        {
            "role": "user",
            "content": "How do I use tool use with the Anthropic Python SDK?",
        }
    ],
)

print("stop_reason:", response_restricted.stop_reason)
print(f"Content blocks: {len(response_restricted.content)}")
print()

for block in response_restricted.content:
    if hasattr(block, "text"):
        print("Claude:", block.text[:500])
        break

## §6. 응답 블록 상세 분석 (Processing Response Blocks)

Web Search 응답에는 일반 Tool Use와 다른 특수 블록이 포함됩니다:

| Block Type | 설명 |
|---|---|
| `server_tool_use` | Claude가 서버에서 실행한 도구 호출 (ServerToolUseBlock) |
| `web_search_tool_result` | 검색 결과 (WebSearchToolResultBlock) |
| `text` | 검색 결과를 종합한 최종 응답 (TextBlock with citations) |

**Citations**: TextBlock의 텍스트 안에 인용 정보가 포함될 수 있습니다.

In [ ]:
# 모든 블록 타입 분석
print("=== Response Block Analysis ===")
print(f"Total blocks: {len(response.content)}")
print()

for i, block in enumerate(response.content):
    block_type = block.type
    print(f"Block {i}: type={block_type}")

    if block_type == "server_tool_use":
        # ServerToolUseBlock: Claude가 서버에서 실행한 검색
        print(f"  name:  {block.name}")
        print(f"  id:    {block.id}")
        if hasattr(block, "input"):
            print(f"  input: {block.input}")

    elif block_type == "web_search_tool_result":
        # WebSearchToolResultBlock: 검색 결과
        print(f"  tool_use_id: {block.tool_use_id}")
        if hasattr(block, "content") and block.content:
            for j, result in enumerate(block.content):
                if hasattr(result, "url"):
                    print(f"  Result {j}: {result.title}")
                    print(f"    URL: {result.url}")
                if j >= 2:  # 처음 3개만 출력
                    print(f"  ... and more results")
                    break

    elif block_type == "text":
        # TextBlock: 최종 텍스트 (인용 포함 가능)
        preview = block.text[:200] + "..." if len(block.text) > 200 else block.text
        print(f"  text: {preview}")

        # Citations 확인
        if hasattr(block, "citations") and block.citations:
            print(f"  citations: {len(block.citations)} found")
            for c in block.citations[:3]:
                if hasattr(c, "url"):
                    print(f"    - {c.url}")

    print()

In [ ]:
# Block 타입별 카운트
from collections import Counter

type_counts = Counter(block.type for block in response.content)

print("Block type counts:")
for btype, count in type_counts.items():
    print(f"  {btype}: {count}")

---

## 정리 (Summary)

### User-Defined Tools vs Built-in Tools

| 구분 | User-Defined | Built-in |
|------|-------------|----------|
| 정의 방식 | `name` + `description` + `input_schema` | `type` + `name` |
| 실행 주체 | **우리 코드**가 실행 | **Claude 서버**가 실행 |
| 결과 전달 | `tool_result` 메시지로 직접 전달 | 자동으로 처리됨 |
| 대화 루프 | 멀티턴 루프 필요 | 단일 호출로 완료 |
| 예시 | `get_current_datetime`, `set_reminder` | `text_editor`, `web_search` |

### 이 시리즈에서 배운 것
1. **NB01**: Tool 함수 작성 + Schema 정의
2. **NB02**: Message blocks 처리 + tool_result 전달
3. **NB03**: Multi-turn 대화 루프 구현
4. **NB04**: 3개 도구를 조합한 완전한 Reminder System
5. **NB05**: Built-in Tools (Text Editor, Web Search)